## Segmentation Skript for training

### 0. Libraries

##### **General**

Pytorch (Torch) backbone for ai setup

Os for loading the images from the directory

json, for saving mean and std

##### **Dataset**

from torch.utils.data import Dataset, for creating a Dataset class to load everything in

from torchvision.io import read_image, reading in the image

from torchvision import tv_tensors, makes the image into a tensor, with .Image or .Mask adds like Metadata

from torchvision.transforms import v2, for the transform operation like resizing and normalizing

from torch.utils.data import DataLoader, for loading data and calulating the mean and std

from torch.utils.data import random_split, for splitting dataset in train,val,test

##### **NN Modules**
import torch.nn as nn, for setting up the architecture for the neural network
import torch.nn.functional as F, for setting up the architecture for the neural network


import torch.optim as optim, for the adam optimizer

from torchmetrics.segmentation import MeanIoU, DiceScore, metrics to track progress

#### **Export**
import torch.onnx, for exporting the model as onnx
import pandas as pd, for creating the csv


In [ ]:

import os
import json
import torch
#Dataset
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torchvision.io import read_image
from torchvision import tv_tensors
from torchvision.transforms import v2
#NN modules
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchmetrics.segmentation import MeanIoU, DiceScore
#export
import torch.onnx
import pandas as pd
import copy

#### 0.1 Paths to Input, Output, Masks...

In [ ]:
IMAGES_FOLDER = "Path/to/the/images"
MASKS_FOLDER = "Path/to/the/masks"
OUTPUT_FOLDER = "./runs"
CSV_TRAINING_PATH = "/trainings_csv.csv"



#### 0.2 Hyperparameter for the model

In [ ]:
img_size_training = (256, 256)
use_new_mean_std = True

# Training parameters
BATCH_SIZE = 16
EARLY_STOPPING_PATIENCE = 12
EPOCHS = 150
LEARNING_RATE = 0.001
THRESHOLD_FOR_PREDICT_PIXEL = 0.5
# Data split
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1




#### 0.3 Intialzing the device, GPU or CPU

In [ ]:

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {DEVICE}')


### 1 Loading the Dataset

### 1.1 Dataset class for loading

In [ ]:
class Satelite_Dataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_only_normalize, transforms):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transforms = transforms #for normalization and resizing
        self.img_only_normalize = img_only_normalize
        self.filenames = os.listdir(img_dir)
        

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        # 1. Build paths
        img_path = os.path.join(self.img_dir, self.filenames[idx]) # image00000
        mask_path = os.path.join(self.mask_dir, "mask_" + self.filenames[idx].split(".")[0] + "_reconstruction.png") # mask_image00000_reconstruction

        # 2. Load image and mask
        img = tv_tensors.Image(read_image(img_path))
        mask = tv_tensors.Mask(read_image(mask_path))
        
        img, mask = self.transforms(img, mask)
        if self.img_only_normalize:
            img = self.img_only_normalize(img)
        
        # Change the 0-255 to 0-1 for BCELoss funciton
        mask = (mask > 0).float()

        # 3. return mask and image
        return img,mask

### 1.2 Calculate mean and std for normalizing

In [ ]:
def calculate_mean_std(dataset):
    loader = DataLoader(dataset, batch_size=32, num_workers=0) 
    
    channels = 3
    cnt = 0
    fst_moment = torch.empty(channels)
    snd_moment = torch.empty(channels)

    for images, _ in loader:
        # images shape: [B, C, H, W]
        b, c, h, w = images.shape
        nb_pixels = b * h * w
        
        sum_ = torch.sum(images, dim=[0, 2, 3])
        sum_of_square = torch.sum(images ** 2, dim=[0, 2, 3])
        
        fst_moment = (cnt * fst_moment + sum_) / (cnt + nb_pixels)
        snd_moment = (cnt * snd_moment + sum_of_square) / (cnt + nb_pixels)
        
        cnt += nb_pixels

    mean = fst_moment
    std = torch.sqrt(snd_moment - fst_moment ** 2)
    
    return mean, std

def new_mean_std():
    basic_transforms = v2.Compose([v2.Resize(img_size_training),v2.ToDtype(torch.float32, scale=True)]) # transfrom to float32 and between 0 and 1
    temp_dataset = Satelite_Dataset(IMAGES_FOLDER, MASKS_FOLDER, transforms=basic_transforms, img_only_normalize=False)
    mean, std = calculate_mean_std(temp_dataset)
    with open('mean_and_std.json', 'w') as f:
        data = {'number_images': len(temp_dataset), 'mean': mean.tolist(), 'std': std.tolist()}
        json.dump(data, f)

def load_stored_mean_std():
    with open('mean_and_std.json', 'r') as f:
        data = json.load(f)
        mean = torch.tensor(data['mean'])
        std = torch.tensor(data['std'])
    return mean, std



#### 1.3 New mean/std or use json values, create datset

In [ ]:
if use_new_mean_std: new_mean_std()
mean, std = load_stored_mean_std()

shared_transforms = v2.Compose([
    v2.Resize(img_size_training, antialias=True),
    v2.ToDtype(torch.float32, scale=True), 
])

img_only_normalize = v2.Normalize(mean=mean, std=std)

training_dataset = Satelite_Dataset(IMAGES_FOLDER, MASKS_FOLDER,img_only_normalize, transforms=shared_transforms)



#### 1.4 Split in Train, Val and Test

In [ ]:
train_dataset, val_dataset, test_dataset = random_split(
    training_dataset, 
    [TRAIN_SPLIT, VAL_SPLIT, 1-TRAIN_SPLIT - VAL_SPLIT],
    generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Training set size: {len(train_dataset)}')
print(f'Validation set size: {len(val_dataset)}')
print(f'Test set size: {len(test_dataset)}')

### 2 Architekture of the Model

#### 2.1 Layers of the architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu1 = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu2 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        return x


class LiteGigaUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base=16):
        super().__init__()

        # -------- Encoder --------
        self.enc1 = ConvBlock(in_channels, base)
        self.down1 = nn.Conv2d(base, base*2, 3, stride=2, padding=1)

        self.enc2 = ConvBlock(base*2, base*2)
        self.down2 = nn.Conv2d(base*2, base*4, 3, stride=2, padding=1)

        self.enc3 = ConvBlock(base*4, base*4)
        self.down3 = nn.Conv2d(base*4, base*8, 3, stride=2, padding=1)

        # -------- Bottleneck --------
        self.bottleneck = ConvBlock(base*8, base*8)

        # -------- Decoder --------
        self.up3_conv = nn.Conv2d(base*8, base*4, 3, padding=1)
        self.dec3 = ConvBlock(base*8, base*4)

        self.up2_conv = nn.Conv2d(base*4, base*2, 3, padding=1)
        self.dec2 = ConvBlock(base*4, base*2)

        self.up1_conv = nn.Conv2d(base*2, base, 3, padding=1)
        self.dec1 = ConvBlock(base*2, base)

        # -------- Output --------
        self.final = nn.Conv2d(base, out_channels, 3, padding=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))

        # Bottleneck
        b = self.bottleneck(self.down3(e3))

        # Decoder
        up3 = F.interpolate(b, scale_factor=2, mode='nearest')
        up3 = self.up3_conv(up3)
        d3  = self.dec3(torch.cat([up3, e3], dim=1))

        up2 = F.interpolate(d3, scale_factor=2, mode='nearest')
        up2 = self.up2_conv(up2)
        d2  = self.dec2(torch.cat([up2, e2], dim=1))

        up1 = F.interpolate(d2, scale_factor=2, mode='nearest')
        up1 = self.up1_conv(up1)
        d1  = self.dec1(torch.cat([up1, e1], dim=1))

        return self.final(d1)

#### 2.2 Loss functions and optimizer

In [ ]:
criterion = nn.BCEWithLogitsLoss()# Binary Cross Entropy Loss for binary segmentation
model = LiteGigaUNet() 
model.to(DEVICE) # Load model to device
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)



#### 2.3 Metrics to track Progress



In [ ]:
dice_metric = DiceScore(num_classes=1).to(DEVICE)
iou_metric = MeanIoU(num_classes=1).to(DEVICE)

### Trainingprocess

#### 3.1 Training on traindata

The validate_model function assesses the models performance on unseen data by calculating both the loss and geometric overlap metrics. After setting the model to evaluation mode and disabling gradient calculations to save memory, the validation loader provides batches of images and masks. The model generates raw logits, which are compared against the ground truth masks to calculate the val_loss.

To evaluate the actual segmentation quality, the logits are passed through a Sigmoid function to obtain pixel-wise probabilities. These are then converted into discrete binary predictions using a Threshold.

In [ ]:
def validate_model(model):
    model.eval()
    val_loss = 0.0
    iou_metric.reset()
    dice_metric.reset()
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            
            outputs = model(images)
            val_loss += criterion(outputs, masks).item()

            preds = (torch.sigmoid(outputs) > THRESHOLD_FOR_PREDICT_PIXEL).long()
            dice_metric.update(preds, masks.long())
            iou_metric.update(preds, masks.long())

    return val_loss/len(val_loader), dice_metric.compute().item(), iou_metric.compute().item()

In [ ]:
def early_stopping(val_loss,modelweights):
    global best_val_loss, current_patience, best_model_wts
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        current_patience = 0
        best_model_wts = copy.deepcopy(model.state_dict())
    else:
        current_patience += 1
        
    return current_patience >= EARLY_STOPPING_PATIENCE          



#### 3.2 Create trainings csv and create ouput folder

In [ ]:
runs = [int(d.split('_')[-1]) for d in os.listdir(OUTPUT_FOLDER) if d.startswith('run_')] if os.path.exists(OUTPUT_FOLDER) else []
current_run_number = max(runs + [0]) + 1
run_dir = os.path.join(OUTPUT_FOLDER, f"run_{current_run_number}")
os.makedirs(run_dir, exist_ok=True)

csv_path = os.path.join(run_dir + CSV_TRAINING_PATH)

def save_csv_training(epoch, train_loss, val_loss, dice, meaniou):
    df = pd.DataFrame({
        'epoch': [epoch],
        'train_loss': [train_loss],
        'val_loss': [val_loss],
        'dice_score': [dice],
        'mean_iou': [meaniou]
    })
    
    file_exists = os.path.isfile(csv_path)
    df.to_csv(csv_path, mode='a', index=False, header=not file_exists)

#### 3.3 Saving the model


In [ ]:
def export_to_onnx(file_name, model_wts):
    model.load_state_dict(model_wts)
    model.eval()
    dummy_input = torch.randn(1, 3, img_size_training[0], img_size_training[1]).to(DEVICE) 
    
    torch.onnx.export(
        model, 
        dummy_input, 
        run_dir + "/" + file_name,
        export_params=True,   # Store the trained weights
        opset_version=11,      # Standard version for compatibility
        input_names=['input'], # Name of the input layer
        output_names=['output'] # Name of the output layer
    )




#### 3.4 Training the Model

In [ ]:
def train_model():
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_wts = None

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for i, (images, masks) in enumerate(train_loader):
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            # Forward & Backward
            outputs = model(images)
            loss = criterion(outputs, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if (i + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{EPOCHS}], Batch [{i+1}/{len(train_loader)}], Current Loss: {loss.item():.4f}")

        # End of epoch summary
        epoch_loss = running_loss / len(train_loader)
        avg_val_loss, dice, meaniou = validate_model(model)

        save_csv_training(epoch+1, epoch_loss, avg_val_loss, dice, meaniou)
        print(f"Epoch [{epoch+1}/{EPOCHS}] completed. Train Loss: {epoch_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Dice: {dice:.4f}, IoU: {meaniou:.4f}")

        # Early stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_wts = copy.deepcopy(model.state_dict())
            export_to_onnx("best_model.onnx", best_model_wts)  # only save when actually improved
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

    export_to_onnx("last_model.onnx", model.state_dict())  # save once at the end

train_model()

#### 3.5 Test on the test part of the data

In [ ]:
def test_model(model, test_loader):
    model.eval()
    iou_metric.reset()
    dice_metric.reset()
        
    with torch.no_grad():
        for images, masks in test_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            
            outputs = model(images)
            preds = (torch.sigmoid(outputs) > THRESHOLD_FOR_PREDICT_PIXEL).long()
            
            dice_metric.update(preds, masks.long())
            iou_metric.update(preds, masks.long())

    final_dice = dice_metric.compute().item()
    final_iou = iou_metric.compute().item()
    
    print("Results on the Test-Set")
    print(f"Dice Score: {final_dice:.4f}")
    print(f"IoU: {final_iou:.4f}")
    
    return final_dice, final_iou
test_model(model,test_loader)